In [ ]:
import numpy
import torch.nn as nn
import torch

#input parameters
n=336 #batch size
l=50 #sequence len
f=4 #number of features dimension (called it c for the convolution examples)

#model parameters
h1 = 32 #first lstm hidden size
h2  = 16 #second lstm hidden size


In [ ]:
#Generate input [336, 50, 4]
rand_arr = torch.rand(n,l,f) 
print(rand_arr.shape)

torch.Size([336, 50, 4])


In [38]:
lstm1 = nn.LSTM(input_size=f, hidden_size=h1, num_layers=1, bidirectional=False, batch_first=True)
out1, _ = lstm1(rand_arr)
print(out1.shape)

torch.Size([336, 50, 32])


In [40]:
lstm2 = nn.LSTM(input_size=h1, hidden_size=h2, num_layers=1, bidirectional=False, batch_first=True)
out2, _ = lstm2(out1)
print(out2.shape)

torch.Size([336, 50, 16])


In [42]:
lstm3 = nn.LSTM(input_size=h2, hidden_size=h2, num_layers=1, bidirectional=False, batch_first=True)
out3, _ = lstm3(out2)
print(out3.shape)


torch.Size([336, 50, 16])


In [43]:
lstm4 = nn.LSTM(input_size=h2, hidden_size=h1, num_layers=1, bidirectional=False, batch_first=True)
out4, _ = lstm4(out3)
print(out4.shape)

torch.Size([336, 50, 32])


In [44]:
dense = nn.Linear(out4.shape[-1], f)
out5 = dense(out4)
print(out5.shape)


torch.Size([336, 50, 4])


In [ ]:
class MVLSTMAutoEncoder(nn.Module):
    def __init__(self, seq_len):
        super(MVLSTMAutoEncoder, self).__init__()
        #input parameters
        self.seq_len = 50 #window length
        self.feature_size = 4 #number of columns
        self.hidden_1 = 32 #starting hidden size
        self.hidden_2 = 16 #ending hidden size (embedding dimension)

        self.encode1 = nn.LSTM(input_size=self.feature_size, hidden_size=self.hidden_1, num_layers=1, bidirectional=False, batch_first=True)
        self.encode2 = nn.LSTM(input_size=self.hidden_1, hidden_size=self.hidden_2, num_layers=1, bidirectional=False, batch_first=True)

        self.decode1 = nn.LSTM(input_size=self.hidden_2, hidden_size=self.hidden_2, num_layers=1, bidirectional=False, batch_first=True)
        self.decode2 = nn.LSTM(input_size=self.hidden_2, hidden_size=self.hidden_1, num_layers=1, bidirectional=False, batch_first=True)

        self.dense = nn.Linear(self.hidden_1, self.feature_size)

    def forward(self, x):
        x, _ = self.encode1(x)
        x, _ = self.encode2(x)
        x, _ = self.decode1(x)
        x, _ = self.decode2(x)
        x = self.dense(x)
        return x